# Amazon Bedrock AgentCore Runtime에서 Strands 에이전트와 Langfuse Observability 사용하기

## 개요

이 노트북에서는 Langfuse 관측성을 연동한 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 배포하는 방법을 살펴봅니다. Amazon Bedrock Claude 모델을 사용하며, OpenTelemetry(OTEL)를 통해 텔레메트리 데이터를 Langfuse로 전송합니다.

## 주요 구성 요소

- **Strands Agents**: 기본 텔레메트리 지원이 포함된 LLM 기반 에이전트 구축용 Python 프레임워크
- **Amazon Bedrock AgentCore Runtime**: AWS에서 에이전트를 호스팅하고 확장하기 위한 관리형 런타임 서비스
- **Langfuse**: OTEL을 통해 트레이스를 수신하는 LLM 애플리케이션용 오픈 소스 관측성 플랫폼
- **OpenTelemetry**: 텔레메트리 데이터를 수집하고 내보내기 위한 업계 표준 프로토콜

## 아키텍처

에이전트는 컨테이너로 패키징되어 호출용 HTTP 엔드포인트를 제공하는 AgentCore Runtime에 배포됩니다. 텔레메트리 데이터는 Strands 에이전트에서 OTEL exporter를 거쳐 Langfuse로 전달되어 모니터링과 디버깅에 사용됩니다. 이 구현에서는 Langfuse를 사용하기 위해 AgentCore의 기본 관측성을 비활성화합니다.

## 사전 요구 사항

- Python 3.10+
- Bedrock 및 AgentCore 권한이 구성된 AWS 자격 증명
- API 키(public key 및 secret key)가 있는 [Langfuse](https://langfuse.com/) 계정
- 로컬에 설치된 Docker
- us-west-2 리전의 Amazon Bedrock Claude 모델에 대한 액세스 권한

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AWS Credentials 구성

## Agent 구현

에이전트 파일(`strands_claude.py`)은 웹 검색 기능을 갖춘 여행 에이전트를 구현합니다. 주요 구성은 다음과 같습니다.
- Strands 텔레메트리 초기화

In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Bedrock 모델 초기화 함수
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Bedrock 모델 초기화
bedrock_model = get_bedrock_model()

# 에이전트의 system prompt 정의
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """올바른 텔레메트리 구성으로 에이전트를 초기화합니다."""

    # 3P 구성으로 Strands 텔레메트리 초기화
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # 에이전트 생성 및 캐시
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # 올바른 구성으로 에이전트 초기화
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용하여 진입점, 앞서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR 리포지토리를 자동으로 생성하도록 starter toolkit을 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다. `bedrock_agentcore_starter_toolkit`으로 에이전트를 구성하면 AgentCore Observability가 기본으로 설정되므로, Langfuse를 사용하려면 아래 설명과 같이 AgentCore Observability 구성을 제거해야 합니다.

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_langfuse_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

## AgentCore Runtime에 배포

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 배포합니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
import base64

# Langfuse 구성
otel_endpoint = "https://us.cloud.langfuse.com/api/public/otel"
langfuse_secret_key = "<langfuse-secret-key>"  # Production에서는 key를 안전하게 저장해야 함
langfuse_public_key = "<langfuse-private-key>"  # Production에서는 key를 안전하게 저장해야 함
langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"


launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",  # model ID 예시
        "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Langfuse OTEL endpoint 사용
        "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Langfuse OTEL auth header 추가
        "DISABLE_ADOT_OBSERVABILITY": "true",
    }
)
launch_result

## 배포 상태 확인

호출하기 전에 런타임이 준비될 때까지 기다립니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

마지막으로 페이로드를 사용하여 AgentCore Runtime을 호출합니다.

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "I'm planning a weekend trip to london. What are the must-visit places and local food I should try?"}
)

In [ ]:
from IPython.display import Markdown, display

display(Markdown("".join(invoke_response["response"])))

## Langfuse에서 트레이스 확인

트레이스를 확인하려면 다음 단계를 따릅니다.
1. https://cloud.langfuse.com 에서 Langfuse 대시보드로 이동합니다.
2. 해당 프로젝트로 이동합니다.
3. "Traces"를 클릭하여 텔레메트리 데이터를 확인합니다.

트레이스에는 다음 정보가 포함됩니다.
- 에이전트 호출 세부 정보
- 도구 호출(웹 검색)
- 지연 시간 및 토큰 사용량을 포함한 모델 상호 작용
- 요청/응답 페이로드

## 리소스 정리(선택 사항)

배포된 리소스를 정리합니다.

In [ ]:
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

## 요약

Langfuse 관측성이 적용된 Strands 에이전트를 Amazon Bedrock AgentCore Runtime에 성공적으로 배포했습니다. 이 구현에서는 다음 내용을 살펴보았습니다.
- Strands 에이전트와 AgentCore Runtime 연동
- Langfuse로 트레이스를 전송하기 위한 OpenTelemetry 구성
- 텔레메트리 구성을 보장하는 올바른 초기화 순서
- SDK 및 boto3 client를 통한 호출

이제 에이전트는 Langfuse를 통해 완전한 관측성을 제공하는 관리형 확장 환경에서 실행됩니다.